# Teste do Handler de Alinhamento de Placas

Este notebook demonstra como usar o módulo `plate_alignment_inspection` para testar o alinhamento e inspeção de placas.

In [ ]:
import sys
import numpy as np
import cv2 as cv
from pathlib import Path

# Adiciona o caminho do módulo ao sistema
project_root = Path.cwd().parent.parent.parent.parent / 'src'
sys.path.insert(0, str(project_root))

# Importa o módulo de inspeção
from open_aoi_services.open_aoi_services.service_product_identification import Service as ProductIdentificationService

print(f"Projeto root: {project_root}")
print(f"Módulos disponíveis: {sys.path[:3]}")

## 1. Carrega o Handler de Alinhamento

In [ ]:
# Importa o módulo de inspeção customizado
sys.path.insert(0, './inspection_development/modules_production')

import plate_alignment_inspection
inspector = plate_alignment_inspection.module

print("✓ Handler carregado com sucesso!")
print(f"\nDocumentação do módulo:\n{plate_alignment_inspection.DOCUMENTATION}")

## 2. Cria Imagens de Teste (Golden e Test)

Vamos criar imagens sintéticas para demonstração

In [ ]:
# Cria uma imagem golden (template) com alguns componentes
def create_golden_image(width=800, height=600):
    """Cria uma imagem golden com componentes para simular uma placa"""
    img = np.ones((height, width, 3), dtype=np.uint8) * 200  # Fundo cinza claro
    
    # Desenha alguns "componentes"
    # Capacitor 1
    cv.rectangle(img, (100, 100), (150, 150), (50, 50, 50), -1)
    cv.putText(img, "C1", (105, 130), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Capacitor 2
    cv.rectangle(img, (200, 100), (250, 150), (50, 50, 50), -1)
    cv.putText(img, "C2", (205, 130), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Resistor 1
    cv.rectangle(img, (350, 120), (400, 140), (100, 100, 100), -1)
    cv.putText(img, "R1", (355, 135), cv.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Chip
    cv.rectangle(img, (500, 80), (600, 180), (50, 50, 50), -1)
    cv.putText(img, "IC1", (530, 135), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    
    # Desenha algumas linhas (trilhas de PCB)
    cv.line(img, (150, 125), (200, 125), (100, 100, 150), 2)
    cv.line(img, (250, 125), (350, 125), (100, 100, 150), 2)
    cv.line(img, (400, 130), (500, 130), (100, 100, 150), 2)
    
    return img

# Cria imagem golden (RGB)
golden_img = create_golden_image()
print(f"Golden Image: {golden_img.shape}")

In [ ]:
# Cria uma imagem test com rotação e pequenas diferenças
def create_test_image_with_issues(golden_img, rotation=0, missing_component=False):
    """Cria imagem de teste com rotação e possíveis defeitos"""
    
    # Aplica rotação
    if rotation != 0:
        h, w = golden_img.shape[:2]
        center = (w // 2, h // 2)
        M = cv.getRotationMatrix2D(center, rotation, 1.0)
        test_img = cv.warpAffine(golden_img, M, (w, h))
    else:
        test_img = golden_img.copy()
    
    # Adiciona ruído
    noise = np.random.normal(0, 5, test_img.shape).astype(np.uint8)
    test_img = cv.add(test_img, noise)
    
    # Se requested, remove um componente (simula defeito)
    if missing_component:
        cv.rectangle(test_img, (200, 100), (250, 150), (200, 200, 200), -1)
    
    return test_img

# Cria 3 cenários de teste
test_img_good = create_test_image_with_issues(golden_img, rotation=2, missing_component=False)
test_img_missing = create_test_image_with_issues(golden_img, rotation=0, missing_component=True)
test_img_rotated = create_test_image_with_issues(golden_img, rotation=5, missing_component=False)

print(f"✓ Imagens de teste criadas")
print(f"  - test_img_good (boa, com pequena rotação)")
print(f"  - test_img_missing (com componente faltando)")
print(f"  - test_img_rotated (com rotação maior)")

In [ ]:
# Visualiza as imagens
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Golden image
axes[0, 0].imshow(cv.cvtColor(golden_img, cv.COLOR_BGR2RGB))
axes[0, 0].set_title('Golden Image (Template)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Test good
axes[0, 1].imshow(cv.cvtColor(test_img_good, cv.COLOR_BGR2RGB))
axes[0, 1].set_title('Test Image (Boa)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Test missing
axes[1, 0].imshow(cv.cvtColor(test_img_missing, cv.COLOR_BGR2RGB))
axes[1, 0].set_title('Test Image (Componente Faltando)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

# Test rotated
axes[1, 1].imshow(cv.cvtColor(test_img_rotated, cv.COLOR_BGR2RGB))
axes[1, 1].set_title('Test Image (Rotada)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("✓ Imagens visualizadas")

## 3. Define Inspection Zones

Vamos criar as zonas de inspeção correspondentes aos componentes

In [ ]:
# Cria as zonas de inspeção (zonas onde queremos verificar componentes)
class MockInspectionZone:
    def __init__(self, zone_id, x, y, width, height, rotation=0):
        self.id = zone_id
        self.rotation = rotation
        # Nested class para simular a estrutura
        self.cc = type('obj', (object,), {
            'stat_left': x,
            'stat_top': y,
            'stat_width': width,
            'stat_height': height
        })

# Define 4 zonas de inspeção (uma para cada componente)
inspection_zones = [
    MockInspectionZone(1, 80, 80, 80, 80),      # Capacitor 1
    MockInspectionZone(2, 180, 80, 80, 80),     # Capacitor 2 (este desaparece no test_missing)
    MockInspectionZone(3, 330, 100, 80, 60),    # Resistor 1
    MockInspectionZone(4, 480, 60, 140, 140),   # Chip IC1
]

print(f"✓ {len(inspection_zones)} zonas de inspeção criadas:")
for zone in inspection_zones:
    print(f"  - Zona {zone.id}: ({zone.cc.stat_left}, {zone.cc.stat_top}) - {zone.cc.stat_width}x{zone.cc.stat_height}")

## 4. Executa Inspeção em 3 Cenários

In [ ]:
# Configuração do ambiente (parâmetros do handler)
environment = {
    "SIMILARITY_THRESHOLD": "0.85",  # 85% de similaridade mínima
    "MAX_FEATURES": "5000",
    "KEEP_PERCENT": "0.2",
    "ALIGNMENT_METHOD": "ORB"
}

print(f"Ambiente configurado:\n{environment}\n")
print("=" * 70)
print("CENÁRIO 1: Imagem Boa (pequena rotação)")
print("=" * 70)

# Converte para RGB (como vem da câmera)
test_rgb = cv.cvtColor(test_img_good, cv.COLOR_BGR2RGB)
template_bgr = golden_img

# Executa inspeção
logs = inspector.process(environment, test_rgb, template_bgr, inspection_zones)

# Mostra resultados
passed_count = 0
for log in logs:
    status = "✓" if log.decision else "✗"
    print(f"{status} {log.description}")
    if log.decision:
        passed_count += 1

print(f"\nResultado: {passed_count}/{len(inspection_zones)} zonas aprovadas")

In [ ]:
print("\n" + "=" * 70)
print("CENÁRIO 2: Imagem com Defeito (componente faltando)")
print("=" * 70)

test_rgb = cv.cvtColor(test_img_missing, cv.COLOR_BGR2RGB)
logs = inspector.process(environment, test_rgb, template_bgr, inspection_zones)

passed_count = 0
for log in logs:
    status = "✓" if log.decision else "✗"
    print(f"{status} {log.description}")
    if log.decision:
        passed_count += 1

print(f"\nResultado: {passed_count}/{len(inspection_zones)} zonas aprovadas")

In [ ]:
print("\n" + "=" * 70)
print("CENÁRIO 3: Imagem Rotada (teste de alinhamento)")
print("=" * 70)

test_rgb = cv.cvtColor(test_img_rotated, cv.COLOR_BGR2RGB)
logs = inspector.process(environment, test_rgb, template_bgr, inspection_zones)

passed_count = 0
for log in logs:
    status = "✓" if log.decision else "✗"
    print(f"{status} {log.description}")
    if log.decision:
        passed_count += 1

print(f"\nResultado: {passed_count}/{len(inspection_zones)} zonas aprovadas")

## 5. Teste com Threshold Diferente

In [ ]:
print("\nTeste com SIMILARITY_THRESHOLD = 0.70 (mais tolerante)\n")

environment_tolerant = {
    "SIMILARITY_THRESHOLD": "0.70",
    "MAX_FEATURES": "5000",
    "KEEP_PERCENT": "0.2",
    "ALIGNMENT_METHOD": "ORB"
}

test_rgb = cv.cvtColor(test_img_missing, cv.COLOR_BGR2RGB)
logs = inspector.process(environment_tolerant, test_rgb, template_bgr, inspection_zones)

for log in logs:
    status = "✓" if log.decision else "✗"
    print(f"{status} {log.description}")

## 6. Teste com Método ECC

In [ ]:
print("\nTeste com ALIGNMENT_METHOD = ECC (mais robusto)\n")

environment_ecc = {
    "SIMILARITY_THRESHOLD": "0.85",
    "MAX_FEATURES": "5000",
    "KEEP_PERCENT": "0.2",
    "ALIGNMENT_METHOD": "ECC"
}

test_rgb = cv.cvtColor(test_img_rotated, cv.COLOR_BGR2RGB)
logs = inspector.process(environment_ecc, test_rgb, template_bgr, inspection_zones)

print(f"Resultado com ECC:")
for log in logs:
    status = "✓" if log.decision else "✗"
    print(f"{status} {log.description}")